# Model training: windows, features and candidates

Questions before writing `model_train.py`:
1. Which labels exist at a given train date, and what is the earliest month a model can be trained?
2. Are the features stable across application months?
3. Which candidate model and feature set does best out of time?
4. Does anything leak?

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import xgboost as xgb

pd.set_option("display.width", 200)

In [2]:
# gold feature store and label store built by the DAG
features = pd.read_parquet("../datamart/gold/feature_store/")
labels = pd.read_parquet("../datamart/gold/label_store/")
features["snapshot_date"] = pd.to_datetime(features["snapshot_date"])
labels["loan_start_date"] = pd.to_datetime(labels["loan_start_date"])
labels["snapshot_date"] = pd.to_datetime(labels["snapshot_date"])
print("features", features.shape, "labels", labels.shape)

features (12500, 68) labels (12500, 6)


## 1. When are labels available?

- A label is `30dpd_6mob`: this is known 6 months after the application (the label store's `snapshot_date`).
- A model trained at month T may only use labels dated before T.

In [3]:
# join features to labels on customer + application month
data = features.merge(
    labels[["Customer_ID", "loan_start_date", "label", "snapshot_date"]].rename(columns={"loan_start_date": "app_month", "snapshot_date": "label_date"}),
    left_on=["Customer_ID", "snapshot_date"], right_on=["Customer_ID", "app_month"], how="inner")
# Spark writes rows in no fixed order; sort so the random split below is the same on every rebuild
data = data.sort_values(["snapshot_date", "Customer_ID"]).reset_index(drop=True)
print("joined:", data.shape[0], "of", features.shape[0], "applications")
print("label date - application month, in months:", ((data["label_date"].dt.year - data["snapshot_date"].dt.year) * 12 + data["label_date"].dt.month - data["snapshot_date"].dt.month).unique())
data.groupby("snapshot_date")["label"].agg(["size", "mean"]).round(3).T

joined: 12500 of 12500 applications
label date - application month, in months: [6]


snapshot_date,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,2023-07-01,2023-08-01,2023-09-01,2023-10-01,...,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01
size,530.000,501.000,506.00,510.000,521.000,517.000,471.000,481.000,454.000,487.000,...,513.000,491.000,498.000,505.000,543.0,493.000,456.000,488.000,515.000,526.000
mean,0.264,0.307,0.32,0.269,0.267,0.265,0.297,0.289,0.262,0.257,...,0.277,0.303,0.311,0.307,0.3,0.331,0.246,0.281,0.272,0.268


In [4]:
# for each candidate train date: how many applications have a label dated before it?
for train_date in ["2024-01-01", "2024-03-01", "2024-06-01", "2024-09-01"]:
    T = pd.Timestamp(train_date)
    available = data[data["label_date"] < T]
    print(train_date, "labelled applications:", len(available),
          "from", available["snapshot_date"].min().date(), "to", available["snapshot_date"].max().date(),
          "=", available["snapshot_date"].nunique(), "months")

2024-01-01 labelled applications: 3085 from 2023-01-01 to 2023-06-01 = 6 months
2024-03-01 labelled applications: 4037 from 2023-01-01 to 2023-08-01 = 8 months
2024-06-01 labelled applications: 5469 from 2023-01-01 to 2023-11-01 = 11 months
2024-09-01 labelled applications: 6961 from 2023-01-01 to 2024-02-01 = 14 months


**Findings:**
- Every application's label is dated exactly 6 months after it, and all 12,500 applications join 1:1 to a label.
- A model trained at T can use applications up to T−7 months.
- Clickstream stops for applications from 2024-07 (A1 finding). A model trained at 2024-09 would score only months without clickstream; there would be nothing "normal" to monitor before the break.
- With 6 + 2 months (8 labelled months), the earliest train date is 2024-03. It then scores 2024-03 to 2024-06 with clickstream and 2024-07 to 2025-01 without. We use 2024-03 with 6 + 2 months.

## 2. Are the features stable across application months?

Population Stability Index (PSI) compares a feature's distribution in a month with its distribution in the training window: < 0.1 stable, 0.1–0.25 moderate drift, > 0.25 significant drift. Bins are the training deciles, missing values get their own bin.

In [5]:
def psi(reference, current):
    """Population Stability Index of current values against reference values.
    Bins are the reference deciles; missing values are their own bin, so a feed that stops shows up."""
    reference, current = pd.Series(reference, dtype=float), pd.Series(current, dtype=float)
    edges = np.unique(np.nanquantile(reference, np.linspace(0.1, 0.9, 9)))

    def proportions(values):
        counts = np.bincount(np.digitize(values.dropna(), edges, right=True), minlength=len(edges) + 1)
        counts = np.append(counts, values.isna().sum())
        return np.clip(counts / len(values), 1e-4, None)  # an empty bin would make log() infinite

    ref, cur = proportions(reference), proportions(current)
    return float(np.sum((cur - ref) * np.log(cur / ref)))

# check on made-up data: same distribution ~0, shifted distribution large, all missing very large
rng = np.random.default_rng(0)
a = rng.normal(0, 1, 5000)
print(round(psi(a, rng.normal(0, 1, 5000)), 3), round(psi(a, rng.normal(1, 1, 5000)), 3), round(psi(a, np.full(500, np.nan)), 1))

0.005 0.954 16.1


In [6]:
# A1's clickstream feature averaged ALL months up to the application; rebuild it from silver to compare windows
clickstream = pd.read_parquet("../datamart/silver/clickstream/")
clickstream["snapshot_date"] = pd.to_datetime(clickstream["snapshot_date"])
apps = features[["Customer_ID", "snapshot_date"]].rename(columns={"snapshot_date": "app_month"})
history = apps.merge(clickstream[["Customer_ID", "snapshot_date", "fe_1"]], on="Customer_ID")
history["months_before"] = (history["app_month"].dt.year - history["snapshot_date"].dt.year) * 12 + history["app_month"].dt.month - history["snapshot_date"].dt.month
history = history[history["months_before"] >= 0]  # never after the application

windows = {
    "all months (A1)": history,
    "last 3 months": history[history["months_before"] < 3],
    "application month": history[history["months_before"] == 0],
}
fe_1 = pd.DataFrame({name: h.groupby(["Customer_ID", "app_month"])["fe_1"].mean() for name, h in windows.items()}).reset_index()
fe_1.groupby("app_month")[list(windows)].std().round(0).iloc[:18:3]

,all months (A1),last 3 months,application month
app_month,,,
2023-01-01,98.0,98.0,98.0
2023-04-01,55.0,62.0,106.0
2023-07-01,37.0,59.0,106.0
2023-10-01,31.0,60.0,99.0
2024-01-01,26.0,58.0,101.0
2024-04-01,25.0,58.0,95.0


In [7]:
# PSI of each window's fe_1 average against the training window (applications 2023-01 to 2023-06)
train_months = fe_1["app_month"] <= "2023-06-01"
fe_1_psi = pd.DataFrame({
    name: fe_1[fe_1["app_month"] >= "2023-07-01"].groupby("app_month")[name].apply(lambda s: psi(fe_1.loc[train_months, name], s))
    for name in windows})
fe_1_psi.round(2).iloc[::3]

,all months (A1),last 3 months,application month
app_month,,,
2023-07-01,0.36,0.03,0.03
2023-10-01,0.74,0.04,0.02
2024-01-01,1.17,0.05,0.03
2024-04-01,1.54,0.04,0.02


**Findings:**
- The `psi` function behaves as expected on made-up data: about 0 for the same distribution, large for a shifted one, very large when every value is missing.
- A1's clickstream average (all months up to the application) is not stable: its spread narrows from std 98 in 2023-01 to 26 in 2024-01 and 25 in 2024-04, because later applicants are averaged over more months. Against the training window its PSI is above 0.25 in every month shown (0.36 in 2023-07, rising to 1.54 in 2024-04), even while the feed is still running. Monitoring would flag it red permanently, so a real break would be invisible.
- A fixed 3-month window keeps its spread at std 58–62 from 2023-04 on, and its PSI stays at 0.05 or below. The application month alone is stable too (PSI ≤ 0.03), but each value is a single noisy month (std about 100).
- So the gold feature store changes to a 3-month window, `fe_*_avg_3m` (`utils/data_processing_gold_table.py`), and is rebuilt for every month. The cells below run on the rebuilt store.

### The rebuilt feature store

In [8]:
# clickstream coverage by application month (A1 finding: none from 2024-07)
data.groupby("snapshot_date")["fe_1_avg_3m"].apply(lambda s: s.notna().mean()).round(2).to_frame("share with clickstream").T

snapshot_date,2023-01-01,2023-02-01,2023-03-01,2023-04-01,2023-05-01,2023-06-01,2023-07-01,2023-08-01,2023-09-01,2023-10-01,...,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01
share with clickstream,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# every feature in the gold store: PSI by application month against the training window
train = data[data["snapshot_date"] <= "2023-06-01"]
feature_cols = [c for c in features.columns if c not in ("Customer_ID", "snapshot_date")]
months = ["2024-03-01", "2024-06-01", "2024-07-01", "2025-01-01"]
feature_psi = pd.DataFrame({m: [psi(train[c], data.loc[data["snapshot_date"] == m, c]) for c in feature_cols] for m in months}, index=feature_cols)
print("features with PSI > 0.25:")
print((feature_psi > 0.25).sum().to_string())
feature_psi.sort_values("2024-06-01", ascending=False).head(8).round(2)

features with PSI > 0.25:
2024-03-01     0
2024-06-01     0
2024-07-01    20
2025-01-01    20


,2024-03-01,2024-06-01,2024-07-01,2025-01-01
fe_9_avg_3m,0.06,0.09,16.11,16.11
fe_20_avg_3m,0.04,0.07,16.11,16.11
fe_18_avg_3m,0.05,0.06,16.11,16.11
fe_5_avg_3m,0.02,0.05,16.11,16.11
fe_10_avg_3m,0.08,0.05,16.11,16.11
fe_15_avg_3m,0.04,0.05,16.11,16.11
fe_3_avg_3m,0.05,0.04,16.11,16.11
Debt_to_Annual_Salary,0.06,0.04,0.02,0.02


**Findings:**
- Every application up to 2024-06 has clickstream features; none from 2024-07 on.
- With `fe_*_avg_3m`, no feature exceeds 0.25 before 2024-07. From 2024-07 the 20 clickstream features are all missing and every one of them is red; the other features stay stable (the highest is below 0.1).

## 3. Candidates: which model and feature set?

The DAG has two training slots (`model_1_automl`, `model_2_automl`): logistic regression and XGBoost, with and without the clickstream features.
 Train/test: applications 2023-01 to 2023-06 (80/20, stratified); OOT: 2023-07 to 2023-08; train date 2024-03-01. Imputation and scaling are fitted on the training split only (prevent train-test contamination).

In [10]:
T = pd.Timestamp("2024-03-01")
train_test = data[(data["snapshot_date"] >= "2023-01-01") & (data["snapshot_date"] <= "2023-06-01")]
oot = data[(data["snapshot_date"] >= "2023-07-01") & (data["snapshot_date"] <= "2023-08-01")]
assert oot["label_date"].max() < T  # every label used is known at the train date
print("train/test", len(train_test), "oot", len(oot), "latest label date used:", oot["label_date"].max().date())

no_click = [c for c in feature_cols if not c.startswith("fe_")]
results = []
for feature_set, cols in [("all features", feature_cols), ("no clickstream", no_click)]:
    X_train, X_test, y_train, y_test = train_test_split(train_test[cols], train_test["label"], test_size=0.2, random_state=88, stratify=train_test["label"])
    imputer = SimpleImputer(strategy="median").fit(X_train)
    scaler = StandardScaler().fit(imputer.transform(X_train))
    prep = lambda X: scaler.transform(imputer.transform(X))
    for name, model in [("logreg", LogisticRegression(C=0.01, max_iter=1000, random_state=88)),
                        ("xgb", xgb.XGBClassifier(n_estimators=50, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=88))]:
        model.fit(prep(X_train), y_train)
        auc = lambda d: roc_auc_score(d["label"], model.predict_proba(prep(d[cols]))[:, 1])
        later = data[data["snapshot_date"] >= T]
        results.append({"features": feature_set, "model": name,
                        "test": roc_auc_score(y_test, model.predict_proba(prep(X_test))[:, 1]), "oot": auc(oot),
                        "scored 2024-03..06": auc(later[later["snapshot_date"] < "2024-07-01"]),
                        "scored 2024-07..2025-01": auc(later[later["snapshot_date"] >= "2024-07-01"])})
pd.DataFrame(results).round(3)

train/test 3085 oot 952 latest label date used: 2024-02-01


,features,model,test,oot,scored 2024-03..06,scored 2024-07..2025-01
0,all features,logreg,0.808,0.838,0.813,0.766
1,all features,xgb,0.827,0.854,0.822,0.797
2,no clickstream,logreg,0.778,0.765,0.768,0.765
3,no clickstream,xgb,0.795,0.806,0.795,0.804


**Findings:**
- XGBoost beats logistic regression on OOT AUC with either feature set, so selecting on OOT AUC picks XGBoost. (`model_train.py` tunes both with a random search; these fixed settings are for comparison only.)
- Clickstream adds about 5 points of OOT AUC for XGBoost (0.854 vs 0.806). It is available at the train date (2024-03), so the model uses it.
- The two right-hand columns use labels that arrive only from 2024-09 on, after the train date. They are shown **for this analysis only**; the pipeline learns them through monitoring. They suggest the clickstream model loses its edge once the feed stops: after 2024-07 it scores 0.797, against 0.804 for the model without clickstream. Monitoring should detect this.
- Decision: train at 2024-03-01 on applications 2023-01 to 2023-06 (train/test) and 2023-07 to 2023-08 (OOT); candidates logistic regression and XGBoost on all gold features; select on OOT AUC.

## 4. Leakage checks

In [ ]:
# temporal: labels dated on or after the train date must not be used
print("labels dated >= train date inside the windows:", (pd.concat([train_test, oot])["label_date"] >= T).sum())
# target: no loan or repayment columns in the feature store
loan_columns = ["loan_id", "loan_start_date", "tenure", "installment_num", "loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance", "mob", "dpd", "label"]
print("loan columns in the feature store:", [c for c in features.columns if c in loan_columns])
# both stores have a snapshot_date, but a label's is 6 months after the application: joining on it matches no application
print("rows matched by a label-date join:", features.merge(labels, on=["Customer_ID", "snapshot_date"]).shape[0])

labels dated >= train date inside the windows: 0
loan columns in the feature store: []
rows matched by a label-date join: 0


**Findings:**
- No label dated on or after the train date is used.
- The feature store holds no loan or repayment columns (`snapshot_date` is the application month, a key).
- Joining on the label date matches 0 rows, so the pipeline joins on `loan_start_date`